<a href="https://colab.research.google.com/github/evakonstantinova/EfficientNet-B0/blob/main/EfficientNet_B0_QuantumNoise_FreeSimulation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# MANUALLY UPLOAD THE TRAINED EFFICIENTNET-B0 CHECKPOINT

from google.colab import files
import os

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError(
        "Please upload exactly one EfficientNet-B0 .pth checkpoint file."
    )

uploaded_filename = next(iter(uploaded))

if not uploaded_filename.lower().endswith(".pth"):
    raise ValueError(
        f"Expected a .pth checkpoint, but received: {uploaded_filename}"
    )

EFFICIENTNET_CHECKPOINT = f"/content/{uploaded_filename}"

if not os.path.exists(EFFICIENTNET_CHECKPOINT):
    raise FileNotFoundError(
        f"Uploaded checkpoint was not found at {EFFICIENTNET_CHECKPOINT}"
    )

print("EfficientNet-B0 checkpoint uploaded successfully.")
print("Filename:", uploaded_filename)
print("Path:", EFFICIENTNET_CHECKPOINT)
print(
    "Size:",
    round(os.path.getsize(EFFICIENTNET_CHECKPOINT) / (1024 ** 2), 2),
    "MB"
)


Saving best_efficientnet_b0_finetuned.pth to best_efficientnet_b0_finetuned.pth
EfficientNet-B0 checkpoint uploaded successfully.
Filename: best_efficientnet_b0_finetuned.pth
Path: /content/best_efficientnet_b0_finetuned.pth
Size: 15.6 MB


In [2]:
# REPRODUCIBILITY SETTINGS

import random
import numpy as np
import torch

SEED = 42

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_global_seed(SEED)

# Improve reproducibility of CUDA operations where applicable.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random seed fixed:", SEED)


Random seed fixed: 42


In [3]:
# DATASET DOWNLOAD AND STRATIFIED SPLIT
# Uses the same 70/15/15 procedure and random_state=42 as the EfficientNet-B0 baseline.

from pathlib import Path
from collections import Counter

import kagglehub
from sklearn.model_selection import train_test_split

path = kagglehub.dataset_download(
    "masoudnickparvar/brain-tumor-mri-dataset"
)

classes = ["glioma", "meningioma", "notumor", "pituitary"]

all_files = []
all_labels = []

for class_name in classes:
    for folder in ["Training", "Testing"]:
        class_path = Path(path) / folder / class_name

        for file_path in class_path.iterdir():
            if file_path.is_file():
                all_files.append(str(file_path))
                all_labels.append(class_name)

train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files,
    all_labels,
    test_size=0.30,
    random_state=42,
    stratify=all_labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

print("Dataset path:", path)
print("Total images:", len(all_files))
print("Overall distribution:", Counter(all_labels))
print("Training:", len(train_files), Counter(train_labels))
print("Validation:", len(val_files), Counter(val_labels))
print("Testing:", len(test_files), Counter(test_labels))

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Dataset path: /kaggle/input/brain-tumor-mri-dataset
Total images: 7200
Overall distribution: Counter({'glioma': 1800, 'meningioma': 1800, 'notumor': 1800, 'pituitary': 1800})
Training: 5040 Counter({'notumor': 1260, 'pituitary': 1260, 'glioma': 1260, 'meningioma': 1260})
Validation: 1080 Counter({'notumor': 270, 'glioma': 270, 'meningioma': 270, 'pituitary': 270})
Testing: 1080 Counter({'meningioma': 270, 'notumor': 270, 'glioma': 270, 'pituitary': 270})


In [4]:
# DATA PREPROCESSING AND DATALOADERS

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class_to_idx = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class BrainTumorDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = Image.open(self.files[idx]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[self.labels[idx]]
        return image, label

train_dataset = BrainTumorDataset(
    train_files,
    train_labels,
    transform=train_transform
)

val_dataset = BrainTumorDataset(
    val_files,
    val_labels,
    transform=val_test_transform
)

test_dataset = BrainTumorDataset(
    test_files,
    test_labels,
    transform=val_test_transform
)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 5040
Validation dataset: 1080
Test dataset: 1080


In [6]:
# PENNYLANE DOWNLOAD
!pip -q install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 94.7 MB/s eta 0:00:00


In [7]:
# IMPORTING LIBRARIES AND CHECKING VERSIONS

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

import pennylane as qml

print("PyTorch version:", torch.__version__)
print("PennyLane version:", qml.__version__)

PyTorch version: 2.11.0+cu128
PennyLane version: 0.45.1


In [8]:
# DEFINING THE QUANTUM CIRCUIT

# DEFINE THE QUANTUM CIRCUIT

N_QUBITS = 4
N_Q_LAYERS = 2

quantum_device = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

@qml.qnode(
    quantum_device,
    interface="torch",
    diff_method="backprop"
)
def quantum_circuit(inputs, weights):

    # ENCODE 4 CLASSICAL FEATURES INTO 4 QUBITS
    qml.AngleEmbedding(
        inputs,
        wires=range(N_QUBITS),
        rotation="Y"
    )

    # APPLY TRAINABLE QUANTUM LAYERS
    qml.StronglyEntanglingLayers(
        weights,
        wires=range(N_QUBITS)
    )

    # MEASURE EACH QUBIT
    return [
        qml.expval(qml.PauliZ(i))
        for i in range(N_QUBITS)
    ]


weight_shapes = {
    "weights": (
        N_Q_LAYERS,
        N_QUBITS,
        3
    )
}

quantum_layer = qml.qnn.TorchLayer(
    quantum_circuit,
    weight_shapes
)

print("Quantum layer created successfully.")
print("Qubits:", N_QUBITS)
print("Quantum layers:", N_Q_LAYERS)

Quantum layer created successfully.
Qubits: 4
Quantum layers: 2


In [9]:
# LOAD THE TRAINED EFFICIENTNET-B0 AND BUILD THE HQNN

# RECREATE THE SAME FOUR-CLASS EFFICIENTNET-B0 ARCHITECTURE.
# weights=None is intentional: no new ImageNet model is loaded here.
efficientnet = models.efficientnet_b0(weights=None)

FEATURE_DIM = efficientnet.classifier[1].in_features

efficientnet.classifier[1] = nn.Linear(
    FEATURE_DIM,
    4
)

# LOAD THE BEST MRI-TRAINED EFFICIENTNET-B0 WEIGHTS.
efficientnet_state = torch.load(
    EFFICIENTNET_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

# strict=True verifies that the checkpoint exactly matches the
# reconstructed four-class EfficientNet-B0 architecture.
efficientnet.load_state_dict(
    efficientnet_state,
    strict=True
)

print("MRI-trained EfficientNet-B0 checkpoint loaded successfully.")
print("EfficientNet feature dimension:", FEATURE_DIM)


class HQNN(nn.Module):

    def __init__(self, trained_efficientnet, quantum_layer):
        super().__init__()

        # REUSE THE MRI-TRAINED EFFICIENTNET-B0 FEATURE EXTRACTOR.
        self.features = trained_efficientnet.features
        self.avgpool = trained_efficientnet.avgpool

        # FREEZE THE COMPLETE EFFICIENTNET-B0 FEATURE EXTRACTOR.
        for param in self.features.parameters():
            param.requires_grad = False

        # NEW TRAINABLE HYBRID CLASSIFICATION HEAD.
        self.feature_reduction = nn.Linear(
            FEATURE_DIM,
            N_QUBITS
        )

        self.quantum_layer = quantum_layer

        self.classifier = nn.Linear(
            N_QUBITS,
            4
        )

    def forward(self, x):

        # FROZEN MRI-TRAINED EFFICIENTNET-B0 FEATURE EXTRACTION.
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        # REDUCE 1280 CLASSICAL FEATURES TO FOUR VALUES.
        x = self.feature_reduction(x)

        # SCALE VALUES INTO A SUITABLE RANGE FOR ANGLE ENCODING.
        x = torch.tanh(x) * torch.pi

        # PENNYLANE default.qubit IS EXECUTED ON CPU.
        x = x.cpu()
        x = self.quantum_layer(x)

        # RETURN QUANTUM OUTPUT TO THE FINAL CLASSIFIER DEVICE.
        classifier_device = next(
            self.classifier.parameters()
        ).device

        x = x.to(classifier_device)

        # FOUR-CLASS OUTPUT.
        x = self.classifier(x)

        return x


hqnn_model = HQNN(
    efficientnet,
    quantum_layer
)

# HARD SAFETY CHECK: THE IMPORTED BACKBONE MUST REMAIN FROZEN.
assert all(
    not p.requires_grad
    for p in hqnn_model.features.parameters()
), "EfficientNet-B0 backbone is not fully frozen."

print(hqnn_model)
print("\nImported MRI-trained EfficientNet-B0 backbone: FROZEN")
print("Trainable components:")
print("  1. feature_reduction")
print("  2. quantum_layer")
print("  3. final classifier")


MRI-trained EfficientNet-B0 checkpoint loaded successfully.
EfficientNet feature dimension: 1280
HQNN(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=Tru

In [10]:
# COUNT AND VERIFY MODEL PARAMETERS

total_params = sum(
    p.numel()
    for p in hqnn_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in hqnn_model.parameters()
    if p.requires_grad
)

quantum_params = sum(
    p.numel()
    for p in hqnn_model.quantum_layer.parameters()
)

backbone_trainable_params = sum(
    p.numel()
    for p in hqnn_model.features.parameters()
    if p.requires_grad
)

assert backbone_trainable_params == 0, (
    "The EfficientNet-B0 backbone must have zero trainable parameters."
)

print(f"Total HQNN parameters: {total_params:,}")
print(f"Trainable HQNN parameters: {trainable_params:,}")
print(f"Quantum trainable parameters: {quantum_params:,}")
print(f"Trainable EfficientNet backbone parameters: {backbone_trainable_params:,}")

print("\nTrainable parameter groups:")
for name, parameter in hqnn_model.named_parameters():
    if parameter.requires_grad:
        print(" ", name, tuple(parameter.shape))


Total HQNN parameters: 4,012,716
Trainable HQNN parameters: 5,168
Quantum trainable parameters: 24
Trainable EfficientNet backbone parameters: 0

Trainable parameter groups:
  feature_reduction.weight (4, 1280)
  feature_reduction.bias (4,)
  quantum_layer.weights (2, 4, 3)
  classifier.weight (4, 4)
  classifier.bias (4,)


In [11]:
# VERIFY THE HQNN FORWARD PASS WITHOUT TOUCHING THE TRAINING LOADER

hqnn_model.eval()

test_images = torch.stack([
    test_dataset[0][0],
    test_dataset[1][0]
])

with torch.no_grad():
    outputs = hqnn_model(test_images)

assert outputs.shape == (2, 4), (
    f"Unexpected HQNN output shape: {outputs.shape}"
)

print("Input shape:", test_images.shape)
print("Output shape:", outputs.shape)
print("Forward-pass check: PASSED")


Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 4])
Forward-pass check: PASSED


In [12]:
# PREPARE THE FROZEN MRI-TRAINED EFFICIENTNET-B0 FEATURE EXTRACTOR

feature_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Feature extraction device:", feature_device)

# MOVE ONLY THE FROZEN MRI-TRAINED EFFICIENTNET PART TO THE AVAILABLE DEVICE
hqnn_model.features = hqnn_model.features.to(feature_device)
hqnn_model.avgpool = hqnn_model.avgpool.to(feature_device)

# KEEP THE FROZEN FEATURE EXTRACTOR IN EVALUATION MODE
hqnn_model.features.eval()
hqnn_model.avgpool.eval()


def extract_efficientnet_features(data_loader):

    all_features = []
    all_labels = []

    # NO GRADIENTS ARE REQUIRED FOR THE FROZEN EFFICIENTNET FEATURE EXTRACTOR
    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(feature_device)

            # EXTRACT EFFICIENTNET-B0 FEATURES
            features = hqnn_model.features(images)
            features = hqnn_model.avgpool(features)
            features = torch.flatten(features, 1)

            # MOVE THE 1280-DIMENSIONAL FEATURES BACK TO CPU
            all_features.append(features.cpu())
            all_labels.append(labels.cpu())

    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_features, all_labels


print("Feature extraction function created successfully.")

Feature extraction device: cuda
Feature extraction function created successfully.


In [13]:
# CONFIGURE THE NOISE-FREE HQNN TRAINING

# THE FROZEN EFFICIENTNET-B0 FEATURE EXTRACTOR MAY USE GPU FOR FEATURE EXTRACTION.
# THE TRAINABLE HYBRID HEAD REMAINS ON CPU FOR PENNYLANE default.qubit.
hqnn_model.feature_reduction = hqnn_model.feature_reduction.cpu()
hqnn_model.quantum_layer = hqnn_model.quantum_layer.cpu()
hqnn_model.classifier = hqnn_model.classifier.cpu()

criterion = nn.CrossEntropyLoss()

# EXPLICITLY OPTIMIZE ONLY THE THREE NEW HYBRID COMPONENTS.
hybrid_trainable_parameters = (
    list(hqnn_model.feature_reduction.parameters())
    + list(hqnn_model.quantum_layer.parameters())
    + list(hqnn_model.classifier.parameters())
)

optimizer = torch.optim.Adam(
    hybrid_trainable_parameters,
    lr=0.001
)

NUM_EPOCHS = 10
best_val_loss = float("inf")

# SAFETY CHECK: NO FROZEN BACKBONE PARAMETER IS PRESENT IN THE OPTIMIZER.
optimizer_parameter_ids = {
    id(parameter)
    for group in optimizer.param_groups
    for parameter in group["params"]
}

backbone_parameter_ids = {
    id(parameter)
    for parameter in hqnn_model.features.parameters()
}

assert optimizer_parameter_ids.isdisjoint(
    backbone_parameter_ids
), "EfficientNet-B0 backbone parameters entered the optimizer."

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Maximum training epochs:", NUM_EPOCHS)
print("Noise-free quantum device: PennyLane default.qubit")
print("EfficientNet-B0 backbone included in optimizer: NO")


Loss function: CrossEntropyLoss
Optimizer: Adam
Learning rate: 0.001
Maximum training epochs: 10
Noise-free quantum device: PennyLane default.qubit
EfficientNet-B0 backbone included in optimizer: NO


In [14]:
# EXTRACT EFFICIENTNET-B0 FEATURES FROM THE VALIDATION DATASET

print("Extracting validation features...")

val_features, val_labels = extract_efficientnet_features(val_loader)

print("Validation feature extraction completed.")
print("Validation features shape:", val_features.shape)
print("Validation labels shape:", val_labels.shape)

Extracting validation features...
Validation feature extraction completed.
Validation features shape: torch.Size([1080, 1280])
Validation labels shape: torch.Size([1080])


In [15]:
# TRAIN THE HYBRID CLASSIFICATION HEAD WITH THE EFFICIENTNET-B0 BACKBONE FROZEN

import time
from sklearn.metrics import f1_score
from torch.utils.data import TensorDataset, DataLoader

# STORE TRAINING HISTORY
train_losses = []
train_accuracies = []
train_f1_scores = []

val_losses = []
val_accuracies = []
val_f1_scores = []

best_val_loss = float("inf")
best_epoch = 0

# START TOTAL TRAINING TIMER
training_start_time = time.time()

for epoch in range(NUM_EPOCHS):

    # FIX THE STOCHASTIC DATA ORDER/AUGMENTATION FOR THIS EPOCH.
    set_global_seed(SEED + epoch)

    print(f"\nEPOCH {epoch + 1}/{NUM_EPOCHS}")
    print("Extracting augmented training features...")

    # EXTRACT NEW TRAINING FEATURES EACH EPOCH
    # THIS PRESERVES RANDOM TRAINING AUGMENTATION
    train_features, train_labels = extract_efficientnet_features(
        train_loader
    )

    # CREATE A FEATURE-LEVEL TRAINING LOADER
    train_feature_dataset = TensorDataset(
        train_features,
        train_labels
    )

    train_feature_loader = DataLoader(
        train_feature_dataset,
        batch_size=32,
        shuffle=True,
        num_workers=0
    )

    # SET THE TRAINABLE HQNN COMPONENTS TO TRAINING MODE
    hqnn_model.feature_reduction.train()
    hqnn_model.quantum_layer.train()
    hqnn_model.classifier.train()

    running_train_loss = 0.0
    train_predictions = []
    train_targets = []

    # TRAIN THE HYBRID CLASSIFIER
    for features, labels in train_feature_loader:

        optimizer.zero_grad()

        # REDUCE 1280 EFFICIENTNET FEATURES TO 4
        outputs = hqnn_model.feature_reduction(features)

        # SCALE THE FOUR FEATURES FOR QUANTUM ANGLE ENCODING
        outputs = torch.tanh(outputs) * torch.pi

        # PASS THE FEATURES THROUGH THE QUANTUM CIRCUIT
        outputs = hqnn_model.quantum_layer(outputs)

        # PRODUCE FOUR-CLASS OUTPUT
        outputs = hqnn_model.classifier(outputs)

        # CALCULATE CLASSIFICATION LOSS
        loss = criterion(outputs, labels)

        # CALCULATE GRADIENTS AND UPDATE TRAINABLE PARAMETERS
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * features.size(0)

        predictions = torch.argmax(outputs, dim=1)

        train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        train_targets.extend(
            labels.cpu().numpy()
        )

    # CALCULATE TRAINING METRICS
    epoch_train_loss = (
        running_train_loss / len(train_feature_dataset)
    )

    epoch_train_accuracy = (
        sum(
            p == t
            for p, t in zip(
                train_predictions,
                train_targets
            )
        )
        / len(train_targets)
    )

    epoch_train_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro"
    )

    # SET THE TRAINABLE HQNN COMPONENTS TO EVALUATION MODE
    hqnn_model.feature_reduction.eval()
    hqnn_model.quantum_layer.eval()
    hqnn_model.classifier.eval()

    # CREATE THE VALIDATION FEATURE LOADER
    val_feature_dataset = TensorDataset(
        val_features,
        val_labels
    )

    val_feature_loader = DataLoader(
        val_feature_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=0
    )

    running_val_loss = 0.0
    val_predictions = []
    val_targets = []

    # EVALUATE ON THE VALIDATION DATASET
    with torch.no_grad():

        for features, labels in val_feature_loader:

            # REDUCE 1280 FEATURES TO 4
            outputs = hqnn_model.feature_reduction(features)

            # SCALE FEATURES FOR QUANTUM ANGLE ENCODING
            outputs = torch.tanh(outputs) * torch.pi

            # PASS THROUGH THE NOISE-FREE QUANTUM CIRCUIT
            outputs = hqnn_model.quantum_layer(outputs)

            # PRODUCE FOUR-CLASS OUTPUT
            outputs = hqnn_model.classifier(outputs)

            loss = criterion(outputs, labels)

            running_val_loss += (
                loss.item() * features.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_targets.extend(
                labels.cpu().numpy()
            )

    # CALCULATE VALIDATION METRICS
    epoch_val_loss = (
        running_val_loss / len(val_feature_dataset)
    )

    epoch_val_accuracy = (
        sum(
            p == t
            for p, t in zip(
                val_predictions,
                val_targets
            )
        )
        / len(val_targets)
    )

    epoch_val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro"
    )

    # STORE EPOCH RESULTS
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)
    train_f1_scores.append(epoch_train_f1)

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_accuracy)
    val_f1_scores.append(epoch_val_f1)

    # SAVE THE CHECKPOINT WITH THE LOWEST VALIDATION LOSS
    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1

        torch.save(
            hqnn_model.state_dict(),
            "/content/BEST_HQNN.pth"
        )

        checkpoint_message = " <-- BEST CHECKPOINT"

    else:
        checkpoint_message = ""

    # PRINT EPOCH RESULTS
    print(
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_accuracy:.4f} | "
        f"Train Macro F1: {epoch_train_f1:.4f}"
    )

    print(
        f"Val Loss:   {epoch_val_loss:.4f} | "
        f"Val Acc:   {epoch_val_accuracy:.4f} | "
        f"Val Macro F1:   {epoch_val_f1:.4f}"
        f"{checkpoint_message}"
    )


# CALCULATE TOTAL TRAINING WALL TIME
training_end_time = time.time()

hqnn_training_time = (
    training_end_time - training_start_time
)

minutes = int(hqnn_training_time // 60)
seconds = int(hqnn_training_time % 60)

print("\nNOISE-FREE HQNN TRAINING COMPLETED")
print("Best checkpoint epoch:", best_epoch)
print(f"Best validation loss: {best_val_loss:.4f}")
print(
    f"Total training wall time: "
    f"{minutes} min {seconds} s"
)


EPOCH 1/10
Extracting augmented training features...
Train Loss: 1.1853 | Train Acc: 0.7774 | Train Macro F1: 0.7775
Val Loss:   1.0692 | Val Acc:   0.8917 | Val Macro F1:   0.8916 <-- BEST CHECKPOINT

EPOCH 2/10
Extracting augmented training features...
Train Loss: 0.8601 | Train Acc: 0.9444 | Train Macro F1: 0.9445
Val Loss:   0.6996 | Val Acc:   0.9315 | Val Macro F1:   0.9314 <-- BEST CHECKPOINT

EPOCH 3/10
Extracting augmented training features...
Train Loss: 0.5044 | Train Acc: 0.9591 | Train Macro F1: 0.9592
Val Loss:   0.4254 | Val Acc:   0.9333 | Val Macro F1:   0.9331 <-- BEST CHECKPOINT

EPOCH 4/10
Extracting augmented training features...
Train Loss: 0.2998 | Train Acc: 0.9631 | Train Macro F1: 0.9631
Val Loss:   0.3144 | Val Acc:   0.9306 | Val Macro F1:   0.9299 <-- BEST CHECKPOINT

EPOCH 5/10
Extracting augmented training features...
Train Loss: 0.2043 | Train Acc: 0.9694 | Train Macro F1: 0.9694
Val Loss:   0.2711 | Val Acc:   0.9361 | Val Macro F1:   0.9357 <-- BEST C

In [16]:
# FINAL TEST OF THE BEST NOISE-FREE HQNN

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# LOAD THE BEST HQNN CHECKPOINT

best_hqnn_state = torch.load(
    "/content/BEST_HQNN.pth",
    map_location="cpu",
    weights_only=True
)

hqnn_model.load_state_dict(
    best_hqnn_state
)

print("Best HQNN checkpoint loaded.")

# VERIFY THAT THE LOADED MODEL STILL USES A FROZEN EFFICIENTNET-B0 BACKBONE.
assert all(
    not p.requires_grad
    for p in hqnn_model.features.parameters()
), "EfficientNet-B0 backbone unexpectedly became trainable."


# CONFIGURE THE MODEL FOR FINAL EVALUATION

evaluation_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


hqnn_model.features = (
    hqnn_model.features.to(evaluation_device)
)

hqnn_model.avgpool = (
    hqnn_model.avgpool.to(evaluation_device)
)

hqnn_model.feature_reduction = (
    hqnn_model.feature_reduction.to(evaluation_device)
)

hqnn_model.classifier = (
    hqnn_model.classifier.to(evaluation_device)
)

hqnn_model.quantum_layer = (
    hqnn_model.quantum_layer.cpu()
)


# SET THE COMPLETE MODEL TO EVALUATION MODE

hqnn_model.eval()


# STORE FINAL TEST OUTPUTS

all_test_targets = []
all_test_predictions = []
all_test_probabilities = []


# EVALUATE THE MODEL ON THE COMPLETE TEST DATASET

with torch.no_grad():

    for images, labels in test_loader:

        # MOVE MRI IMAGES AND LABELS TO THE CLASSICAL DEVICE

        images = images.to(
            evaluation_device
        )

        labels = labels.to(
            evaluation_device
        )

        # RUN THE COMPLETE HQNN

        outputs = hqnn_model(
            images
        )

        # CONVERT LOGITS TO CLASS PROBABILITIES

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        # SELECT THE CLASS WITH THE HIGHEST LOGIT

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        # STORE TARGET LABELS

        all_test_targets.extend(
            labels.cpu().numpy()
        )

        # STORE PREDICTED LABELS

        all_test_predictions.extend(
            predictions.cpu().numpy()
        )

        # STORE CLASS PROBABILITIES

        all_test_probabilities.extend(
            probabilities.cpu().numpy()
        )


# CONVERT RESULTS TO NUMPY ARRAYS

all_test_targets = np.array(
    all_test_targets
)

all_test_predictions = np.array(
    all_test_predictions
)

all_test_probabilities = np.array(
    all_test_probabilities
)


# CALCULATE FINAL TEST METRICS

test_accuracy = accuracy_score(
    all_test_targets,
    all_test_predictions
)

test_precision = precision_score(
    all_test_targets,
    all_test_predictions,
    average="macro"
)

test_recall = recall_score(
    all_test_targets,
    all_test_predictions,
    average="macro"
)

test_f1 = f1_score(
    all_test_targets,
    all_test_predictions,
    average="macro"
)

test_roc_auc = roc_auc_score(
    all_test_targets,
    all_test_probabilities,
    multi_class="ovr",
    average="macro"
)


# PRINT FINAL TEST RESULTS

print(
    "\nFINAL NOISE-FREE HQNN TEST RESULTS"
)

print(
    f"Accuracy:        {test_accuracy:.4f}"
)

print(
    f"Macro Precision: {test_precision:.4f}"
)

print(
    f"Macro Recall:    {test_recall:.4f}"
)

print(
    f"Macro F1-score:  {test_f1:.4f}"
)

print(
    f"Macro ROC-AUC:   {test_roc_auc:.4f}"
)


# PRINT CLASSIFICATION RESULTS FOR EACH TUMOR CLASS

print(
    "\nCLASSIFICATION REPORT"
)

print(
    classification_report(
        all_test_targets,
        all_test_predictions,
        target_names=[
            "Glioma",
            "Meningioma",
            "No Tumor",
            "Pituitary"
        ],
        digits=4
    )
)

Best HQNN checkpoint loaded.

FINAL NOISE-FREE HQNN TEST RESULTS
Accuracy:        0.9315
Macro Precision: 0.9313
Macro Recall:    0.9315
Macro F1-score:  0.9309
Macro ROC-AUC:   0.9888

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Glioma     0.9118    0.9185    0.9151       270
  Meningioma     0.9206    0.8593    0.8889       270
    No Tumor     0.9336    0.9889    0.9604       270
   Pituitary     0.9593    0.9593    0.9593       270

    accuracy                         0.9315      1080
   macro avg     0.9313    0.9315    0.9309      1080
weighted avg     0.9313    0.9315    0.9309      1080



In [17]:
# DOWNLOAD THE FINAL BEST HQNN CHECKPOINT TO THE COMPUTER

from google.colab import files
import os

HQNN_CHECKPOINT = "/content/BEST_HQNN.pth"

if not os.path.exists(HQNN_CHECKPOINT):
    raise FileNotFoundError(
        "BEST_HQNN.pth was not found. Run the HQNN training first."
    )

print("Final HQNN checkpoint ready.")
print("Path:", HQNN_CHECKPOINT)
print(
    "Size:",
    round(os.path.getsize(HQNN_CHECKPOINT) / (1024 ** 2), 2),
    "MB"
)
print("Downloading BEST_HQNN.pth...")

files.download(HQNN_CHECKPOINT)


Final HQNN checkpoint ready.
Path: /content/BEST_HQNN.pth
Size: 15.59 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>